In [ ]:
Data Science Internship – February 2026  
NLP Assignment 3: Chatbot using Transformers  

Name: Hamsini S Y  

In [4]:
!pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [6]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# FIX: Set pad token
tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model loaded successfully!


In [7]:
def chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")
    
    chat_history_ids = None

    while True:
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a nice day.")
            break

        # Tokenize input with attention mask
        inputs = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
        new_input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']

        # Maintain conversation history
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
            attention_mask = torch.cat([torch.ones_like(chat_history_ids), attention_mask], dim=-1)
        else:
            bot_input_ids = new_input_ids

        # Generate response (FIX APPLIED)
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )

        # Decode response
        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print("Chatbot:", response, "\n")

In [8]:
chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?



You:  Hello


Chatbot: Hey there 



You:  what is artificial intelligence 


Chatbot: It's a new thing. 



...

Chatbot: I didn't create it, but I'm a part of the development team. 



You:  Thank you


Chatbot: I'm very excited to work on python. 



You:  exit


Chatbot: Goodbye! Have a nice day.


In [ ]:
This chatbot is built using a pre-trained transformer model (DialoGPT) from Hugging Face.

The chatbot:
- Accepts user input
- Processes text using tokenizer
- Generates responses using transformer model
- Maintains conversation history
- Stops when user types "exit" or "quit"

The attention mask is explicitly passed to avoid ambiguity between padding and actual tokens, ensuring better performance.

Conclusion:
Transformer-based models enable powerful conversational AI systems capable of generating human-like responses.